# Agentes de IA con LangChain y Gemini — Fundamentos

## ¿De qué trata este notebook?

Este notebook te enseña a construir **agentes de IA**. Es la introducción al kit completo: cómo crear un agente, darle herramientas, controlar el modelo, formatear respuestas y construir conversaciones.

Usa dos herramientas:

- **Gemini**: el modelo de lenguaje de Google (el "cerebro" que entiende y genera texto)
- **LangChain**: un framework (una caja de herramientas) que facilita conectar ese cerebro con acciones del mundo real

Un **agente** es simplemente un programa de IA que puede:
1. Recibir una pregunta o instrucción en lenguaje normal
2. Decidir qué herramientas necesita para responder
3. Usar esas herramientas
4. Devolver una respuesta

La diferencia con un chatbot simple es que **el agente puede hacer cosas**, no solo hablar.

---

## Temas que cubre este notebook

| Sección | Qué aprenderás |
|---------|----------------|
| Modelo estático | Crear el agente más básico posible |
| Modelo dinámico | Hacer que el agente cambie de modelo según la situación |
| System prompt | Darle al agente una personalidad o rol fijo |
| Stream | Ver la respuesta llegar en tiempo real, palabra a palabra |
| Structured output | Hacer que el agente devuelva datos en un formato exacto |
| Tools | Darle al agente herramientas que puede usar |
| Tools dinámicas | Cambiar qué herramientas tiene disponibles según el contexto |
| Mensajes | Construir conversaciones con roles (sistema, usuario, IA) |

## Paso 0 — Instalación y carga de la clave de API

Antes de hacer nada, necesitamos dos cosas:

1. **Las librerías** instaladas (el código de terceros que vamos a usar)
2. **La API Key de Gemini** — una contraseña que le dice a Google que somos nosotros los que estamos haciendo las peticiones

La API Key se guarda en un lugar seguro llamado **Secrets** (en Google Colab) y se recupera con `userdata.get(...)`. Así nunca aparece escrita directamente en el código, que podría ser visto por otros.

In [1]:
import os
from dotenv import load_dotenv

# Lee el archivo .env si existe
load_dotenv(dotenv_path="C:/Users/Oscar/OneDrive - FM4/Escritorio/EVOLVE/Data Science/EVOLVE/Victor_Prior_IA_Generativa/Proyecto_Modulo_IAGen/.env")

# En Google Colab usa: from google.colab import userdata; GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
# En local con .env usa: GOOGLE_API_KEY = os.getenv("Gemini_API_Key_001")
GOOGLE_API_KEY = os.getenv("Gemini_API_Key_001")
print("✅ API Key cargada")

✅ API Key cargada


In [ ]:
# Descomenta la línea de abajo si necesitas instalar las librerías por primera vez
# !pip install langchain-core langchain

### Importar las herramientas que vamos a usar

Esta celda carga todas las piezas que necesitamos del kit de herramientas de LangChain:

| Lo que importamos | Para qué sirve |
|-------------------|----------------|
| `ChatGoogleGenerativeAI` | La conexión con el modelo Gemini de Google |
| `ChatPromptTemplate` | Una plantilla para construir prompts con variables |
| `MessagesPlaceholder` | Un espacio reservado para insertar el historial de mensajes |
| `HumanMessage` | Representa un mensaje del usuario en la conversación |
| `AIMessage` | Representa un mensaje de respuesta de la IA |
| `SystemMessage` | Representa las instrucciones iniciales del sistema |
| `create_agent` | La función que crea el agente de IA |

In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

from langchain.agents import create_agent

---

## Sección 1 — Modelo estático: el agente más básico

### ¿Qué es un modelo estático?

"Estático" significa que el agente siempre usa el mismo modelo de IA, con los mismos parámetros, sin importar lo que pase en la conversación. Es como tener siempre al mismo empleado haciendo el mismo trabajo con las mismas instrucciones.

### ¿Qué es `temperature`?

La temperatura controla qué tan creativo o predecible es el modelo:
- **Temperatura 0**: respuestas muy consistentes y predecibles. Ideal para tareas precisas (extraer datos, clasificar).
- **Temperatura 0.5**: equilibrio entre precisión y variedad.
- **Temperatura 1.0**: respuestas muy creativas e impredecibles. Ideal para escritura creativa.

In [10]:
# Definimos qué modelo de Gemini usar
# "gemini-2.5-flash-lite" es la versión más ligera y rápida (y más económica)
MODEL = "gemini-2.5-flash-lite"

In [12]:
# Creamos la conexión con el modelo de Gemini
# temperature=0.5 → equilibrio entre precisión y variedad en las respuestas
# google_api_key=API_KEY → usamos la clave que cargamos antes
llm = ChatGoogleGenerativeAI(model=MODEL, temperature=0.5, google_api_key=GOOGLE_API_KEY)

In [13]:
# Creamos el agente
# tools=None → este agente no tiene herramientas, solo puede conversar
# Es el agente más básico posible
agent = create_agent(llm, tools=None)

---

## Sección 2 — Modelo dinámico: cambiar de modelo según el contexto

### ¿Por qué cambiar de modelo?

No todas las conversaciones son igual de complejas. Una pregunta simple ("¿qué tiempo hace?") no necesita el modelo más potente. Una conversación técnica larga sí puede necesitarlo.

El **modelo dinámico** funciona como una empresa con empleados de diferentes niveles:
- Para tareas sencillas → envías al junior (más rápido y barato)
- Para tareas complejas o largas → llamas al senior (más potente pero más caro)

### ¿Qué es un `middleware`?

Un middleware es un intermediario que se ejecuta **antes** de que el modelo reciba la petición. Puede modificar la petición, cambiar el modelo a usar, filtrar herramientas, etc. Es como un filtro o una puerta de entrada con lógica.

In [14]:
# Importamos las herramientas para crear el middleware de selección de modelo
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse

In [16]:
# Creamos dos versiones del modelo:
# basic_model → el modelo ligero (para conversaciones cortas)
# advanced_model → el modelo más potente (para conversaciones largas o complejas)
# Ambos con temperature=0.9 → respuestas más variadas y creativas
basic_model = ChatGoogleGenerativeAI(model=MODEL, temperature=0.9, google_api_key=GOOGLE_API_KEY)
advanced_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.9, google_api_key=GOOGLE_API_KEY)

In [17]:
# Este decorador (@wrap_model_call) convierte esta función en un middleware
# que se ejecuta cada vez que el agente va a llamar al modelo
@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity"""
    # Contamos cuántos mensajes hay en la conversación
    message_count = len(request.state["messages"])

    # Si hay más de 10 mensajes, la conversación es larga → usamos el modelo avanzado
    if message_count > 10:
        model = advanced_model
    # Si hay 10 o menos mensajes → usamos el modelo básico (más rápido y barato)
    else:
        model = basic_model

    # Ejecutamos la petición con el modelo elegido
    return handler(request.override(model=model))

In [18]:
# Creamos el agente con el middleware de selección dinámica de modelo
# middleware=[dynamic_model_selection] → aplica nuestra lógica de selección antes de cada llamada
agent = create_agent(
    model=basic_model,
    tools=[],
    middleware=[dynamic_model_selection]
    )

---

## Sección 3 — System prompt: darle una personalidad al agente

### ¿Qué es un system prompt?

El **system prompt** son las instrucciones que el agente recibe antes de cualquier conversación. Es como el briefing que le das a un empleado antes de atender al cliente:

- "Eres un asistente especializado en literatura"
- "Responde siempre en español"
- "No des información sobre precios"
- "Sé formal y profesional"

El usuario no ve estas instrucciones, pero el modelo las sigue en todas sus respuestas.

### ¿Qué hace el código de abajo?

Crea un agente con un rol fijo (especialista en literatura), le añade el middleware de modelo dinámico, y luego le hace una pregunta sobre El Quijote para ver cómo responde.

In [19]:
# Creamos el agente con system prompt
# system_prompt → instrucciones fijas que el modelo siempre tiene en cuenta
# En este caso: el agente se comportará como especialista en literatura
agent = create_agent(
    model=advanced_model,
    tools=[],
    system_prompt = "You are an AI assistant specialized in literarys",
    middleware=[dynamic_model_selection]
    )

In [20]:
# Enviamos un mensaje al agente
# El mensaje viene en formato lista de diccionarios con "role" y "content"
# "role": "user" → indica que es el usuario quien habla
result = agent.invoke(
    {"messages": [{"role": "user", "content": "El Quijote"}]}
)

In [21]:
# Mostramos el resultado completo (incluye toda la estructura de mensajes)
result

{'messages': [HumanMessage(content='El Quijote', additional_kwargs={}, response_metadata={}, id='5baebc4a-e0cf-45d9-8d68-e887a150abb6'),
  AIMessage(content='Ah, **Don Quixote**! A monumental and beloved work in Spanish literature, and indeed, world literature. It\'s a novel that has resonated with readers for centuries and continues to be studied, adapted, and cherished.\n\nTo give you a comprehensive answer, I need a little more direction. What about Don Quixote are you interested in? For example, would you like to know about:\n\n*   **The Author:** Miguel de Cervantes Saavedra and his life.\n*   **The Plot:** A summary of the adventures of Don Quixote and Sancho Panza.\n*   **Key Themes:** Such as idealism vs. realism, madness, heroism, illusion vs. reality, literature itself, and social commentary.\n*   **Major Characters:** Don Quixote, Sancho Panza, Dulcinea del Toboso, and others.\n*   **Literary Significance:** Its impact on the novel form, its genre (parody, picaresque, romanc

---

## Sección 4 — Stream: ver la respuesta en tiempo real

### ¿Qué es el streaming?

Cuando consultas a ChatGPT y ves las palabras aparecer una a una en lugar de esperar a que llegue el texto completo, eso es **streaming**.

Sin streaming: el agente procesa todo → esperas → recibes la respuesta completa de golpe.

Con streaming: el agente genera token a token (palabra a palabra) → tú ves cada pieza según llega → la experiencia es más fluida.

### ¿Cuándo usarlo?

- En interfaces de usuario donde el usuario espera ver la respuesta aparecer (chatbots, asistentes)
- Cuando las respuestas son largas y no quieres que el usuario espere en silencio

### ¿Qué hace el código de abajo?

Crea una herramienta de tiempo (`get_weather`) y un agente que la usa. Luego pregunta por el tiempo en Madrid y muestra cómo llega la respuesta paso a paso.

In [22]:
# Esta función simula una consulta del tiempo
# En un sistema real, haría una petición a una API meteorológica
# Aquí simplemente devuelve un texto fijo para demostrar el concepto
def get_weather(city: str) -> str:
    """ Get weather for a given city """
    return f"It is always sunny in {city}"

In [23]:
# Creamos un agente con la herramienta del tiempo
# tools=[get_weather] → el agente puede usar esta función cuando la necesite
agent = create_agent(
    model = basic_model,
    tools = [get_weather]
)

In [24]:
# agent.stream() en lugar de agent.invoke() → recibe la respuesta en fragmentos
# stream_mode="updates" → muestra cada actualización del estado del agente
# El bucle for itera sobre cada fragmento según llega
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in Madrid?"}]},
    stream_mode="updates",
):
    # chunk es un diccionario con el paso actual y sus datos
    for step, data in chunk.items():
        print(f"step: {step}")  # Nombre del paso (ej: "agent", "tools")
        print(f"content: {data['messages'][-1].content_blocks}")  # Contenido del fragmento

step: model
content: [{'type': 'tool_call', 'id': '98dea581-3992-4a8a-b5f7-e28b7ee8edb7', 'name': 'get_weather', 'args': {'city': 'Madrid'}}]
step: tools
content: [{'type': 'text', 'text': 'It is always sunny in Madrid'}]
step: model
content: []


---

## Sección 5 — Structured output: respuestas con formato garantizado

### El problema que resuelve

Por defecto, un LLM responde con texto libre. Si le pides "extrae el nombre y email de este texto", puede responder:
- "El nombre es Juan García y su email es juan@mail.com"
- "Nombre: Juan García. Email: juan@mail.com"
- "He encontrado la siguiente información: nombre=Juan García, correo=juan@mail.com"

Cada vez puede ser diferente. Para usarlo en código, necesitas parsear el texto manualmente.

**Structured output** resuelve esto: defines exactamente qué campos quieres y de qué tipo, y el modelo te devuelve siempre eso, de forma consistente.

### ¿Qué es Pydantic?

Pydantic es una librería de Python para definir estructuras de datos con tipos y validaciones. Aquí la usamos para decirle al modelo: "quiero que la respuesta siempre tenga exactamente estos campos".

In [25]:
# Importamos las herramientas de Pydantic para definir estructuras de datos
from pydantic import BaseModel, Field

In [26]:
# Definimos el esquema (la "plantilla") que queremos que siga la respuesta
# BaseModel → indica que esta clase es un esquema de datos de Pydantic
class ConctactInfo(BaseModel):
    """Contact information for a person"""
    # Cada campo tiene un nombre, un tipo (str = texto) y una descripción
    # La descripción le dice al modelo qué debe poner en ese campo
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

In [27]:
# Creamos el agente con el formato de respuesta estructurada
# response_format=ConctactInfo → el agente devolverá SIEMPRE un objeto con los campos name, email y phone
agent = create_agent(
    model = basic_model,
    response_format=ConctactInfo
)

In [28]:
# Le pedimos al agente que extraiga información de contacto de un texto libre
# El texto contiene: nombre, email y teléfono mezclados en una cadena
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (+34) 666 66 66 66"}]
})

In [29]:
# El resultado estructurado está en result["structured_response"]
# No es texto libre: es un objeto Python con atributos .name, .email, .phone
print(result["structured_response"])

name='John Doe' email='john@example.com' phone='(+34) 666 66 66 66'


---

## Sección 6 — Tools: darle herramientas al agente

### ¿Qué son las tools (herramientas)?

Las tools son funciones de Python que el agente puede decidir llamar cuando las necesita.

Sin tools, el agente solo puede generar texto. Con tools, puede:
- Buscar en internet
- Consultar una base de datos
- Llamar a una API externa
- Ejecutar cálculos
- Enviar emails

El agente lee la descripción de cada tool (el texto entre `"""`), decide cuándo necesita usarla, y la llama con los argumentos correctos.

### El decorador `@tool`

`@tool` es lo que convierte una función normal de Python en una herramienta que el agente puede usar. Sin él, el agente no "ve" la función.

In [30]:
# Importamos el decorador @tool de LangChain
from langchain.tools import tool

In [31]:
# Definimos dos herramientas con @tool
# La descripción entre """ """ es CRÍTICA: el agente la lee para decidir cuándo usar cada herramienta

@tool
def search(query: str) -> str:
    """Search for information"""  # ← El agente lee esto para saber qué hace esta función
    return f"Results for: {query}"  # Simulación de búsqueda

@tool
def get_weather(city: str) -> str:
    """ Get weather for a given city """  # ← El agente lee esto para saber qué hace esta función
    return f"It is always sunny in {city}"  # Simulación de consulta del tiempo

In [32]:
# Creamos el agente con las dos herramientas
# El agente decidirá automáticamente qué herramienta usar según la pregunta del usuario
agent = create_agent(llm, tools=[search, get_weather])

---

## Sección 7 — Tools dinámicas: filtrar herramientas según el contexto

### ¿Por qué controlar qué herramientas tiene el agente?

Imagina un banco con un empleado de atención al cliente. Las funciones que puede hacer ese empleado deberían depender de quién es el cliente:
- Cliente no autenticado → solo puede ver información pública
- Cliente autenticado → puede ver su saldo, hacer transferencias
- Empleado interno → acceso completo

El código de abajo implementa exactamente eso: un middleware que filtra las herramientas disponibles según si el usuario está autenticado y cuántos mensajes lleva la conversación.

**Nota:** Este ejemplo usa herramientas (`public_search`, `private_search`, `advanced_search`) que no están definidas en este notebook. Es una demostración del patrón, no código ejecutable directamente.

In [ ]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

# Middleware que filtra las herramientas según el estado de la conversación
@wrap_model_call
def state_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Filter tools based on conversation State."""
    # Leemos el estado: ¿está el usuario autenticado? ¿cuántos mensajes lleva?
    state = request.state
    is_authenticated = state.get("authenticated", False)  # False si no hay dato de autenticación
    message_count = len(state["messages"])

    # Si NO está autenticado → solo puede usar herramientas públicas (las que empiezan con "public_")
    if not is_authenticated:
        tools = [t for t in request.tools if t.name.startswith("public_")]
        request = request.override(tools=tools)
    # Si está autenticado pero la conversación es nueva (menos de 5 mensajes)
    # → puede usar casi todo, pero no la búsqueda avanzada todavía
    elif message_count < 5:
        tools = [t for t in request.tools if t.name != "advanced_search"]
        request = request.override(tools=tools)

    return handler(request)

# NOTA: Este código requiere que public_search, private_search y advanced_search
# estén definidos previamente. Es un patrón de diseño, no ejecutable directamente.
# agent = create_agent(
#     model="gpt-4.1",
#     tools=[public_search, private_search, advanced_search],
#     middleware=[state_based_tools]
# )

---

## Sección 8 — Mensajes: construir conversaciones con roles

### ¿Cómo estructura LangChain una conversación?

Toda conversación con un LLM tiene tres tipos de mensajes:

| Tipo | Quién habla | Para qué |
|------|-------------|----------|
| **System** | El sistema (nosotros) | Instrucciones fijas que el modelo sigue siempre |
| **Human** | El usuario | Las preguntas o instrucciones del usuario |
| **AI** | El modelo | Las respuestas generadas |

Un `ChatPromptTemplate` es una plantilla con variables (`{idioma}`, `{texto}`) que se rellenan en tiempo real. Esto permite reutilizar la misma estructura para distintos inputs.

### ¿Qué hace el código de abajo?

Crea un traductor que convierte texto al idioma que le indiques. La "chain" (`chat_prompt | chat`) encadena la plantilla con el modelo: primero formatea el prompt, luego se lo envía al modelo.

In [52]:
# Definimos una plantilla de conversación con dos variables:
# {idioma} → el idioma al que traducir
# {texto}  → el texto que hay que traducir
chat_prompt = ChatPromptTemplate([
    ("system", "Eres un traductor del español al {idioma} muy preciso"),  # Rol del sistema
    ("human", "{texto}")  # El texto que el usuario quiere traducir
])

# Rellenamos las variables de la plantilla con valores concretos
# Esto genera la lista de mensajes que se enviará al modelo
mensajes = chat_prompt.format_messages(texto="Hola mundo, te quiero mucho", idioma="francés")

In [53]:
# Mostramos qué tipo de mensaje es cada uno y su contenido
# Veremos: SystemMessage con las instrucciones y HumanMessage con "Hola mundo"
for m in mensajes:
    print(f"{type(m)}: {m.content}")

<class 'langchain_core.messages.system.SystemMessage'>: Eres un traductor del español al francés muy preciso
<class 'langchain_core.messages.human.HumanMessage'>: Hola mundo, te quiero mucho


In [54]:
# Creamos el modelo de chat
chat = ChatGoogleGenerativeAI(model=MODEL, temperature = 0.5, google_api_key=GOOGLE_API_KEY)

In [55]:
# Creamos la cadena (chain) que conecta la plantilla con el modelo
# El operador | (pipe) encadena componentes: lo que sale de chat_prompt entra en chat
chain = chat_prompt | chat

In [56]:
# Ejecutamos la cadena con valores concretos
# Internamente: formatea el prompt → envía al modelo → devuelve respuesta
resultado = chat.invoke(mensajes)

In [57]:
# Mostramos el resultado completo
# .content contiene el texto de la traducción
list(resultado)

[('content', "Bonjour le monde, je t'aime beaucoup"),
 ('additional_kwargs', {}),
 ('response_metadata',
  {'finish_reason': 'STOP',
   'model_name': 'gemini-2.5-flash-lite',
   'safety_ratings': [],
   'model_provider': 'google_genai'}),
 ('type', 'ai'),
 ('name', None),
 ('id', 'lc_run--019dd565-41f9-78f1-a3c4-ab2baad20877-0'),
 ('tool_calls', []),
 ('invalid_tool_calls', []),
 ('usage_metadata',
  {'input_tokens': 17,
   'output_tokens': 9,
   'total_tokens': 26,
   'input_token_details': {'cache_read': 0}})]

---

## Resumen del notebook

| Concepto aprendido | Cuándo usarlo |
|--------------------|---------------|
| **Modelo estático** | Prototipado rápido, casos simples |
| **Modelo dinámico** | Cuando quieres optimizar coste según complejidad |
| **System prompt** | Dar un rol o personalidad fija al agente |
| **Stream** | Interfaces de usuario que muestran la respuesta en tiempo real |
| **Structured output** | Cuando necesitas datos en formato exacto para usar en código |
| **Tools** | Cuando el agente necesita hacer cosas, no solo hablar |
| **Tools dinámicas** | Control de acceso y seguridad según contexto |
| **Mensajes / Chains** | Construir pipelines reutilizables con plantillas |